# Step 1: Concatenate Garmin sleep data
**Author**: maxk678 | **Data Source:** [Garmin Data Request](https://www.garmin.com/en-US/account/datamanagement/exportdata)

This notebook loads all raw Garmin sleep JSON Files, combines them into a single table, and saves as a CSV.



---
# Imports and Setup

In [11]:
import json
import glob
import pandas as pd

---
## 1. Find sleep files

Using `*` as a wildcard since files follow the same naming convention

Update the path to your local raw data folder

In [12]:
sleep_files = sorted(glob.glob(
    r"C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\data\raw\JSONs\DI_CONNECT\Sleep\*sleepData.json"
))

print(f"Found {len(sleep_files)} files.")

Found 9 files.


---
## 2. Load each file and combine into one list

Looping through every file (a JSON array) and adding its records to `all_records`. 

In [13]:
all_records = []

for filepath in sleep_files:
    with open(filepath) as f:
        records = json.load(f)
    all_records.extend(records)

print(f"\nTotal records loaded: {len(all_records)}")


Total records loaded: 800


---
## 3. Convert to a DataFrame

In [14]:
# Convert into a DataFrame
df = pd.DataFrame(all_records)

print(f"\nRaw shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


Raw shape: (800, 18)
Columns: ['sleepStartTimestampGMT', 'sleepEndTimestampGMT', 'calendarDate', 'sleepWindowConfirmationType', 'napList', 'deepSleepSeconds', 'lightSleepSeconds', 'remSleepSeconds', 'awakeSleepSeconds', 'unmeasurableSeconds', 'averageRespiration', 'lowestRespiration', 'highestRespiration', 'retro', 'awakeCount', 'avgSleepStress', 'sleepScores', 'restlessMomentCount']


---
## 4. Flatten `sleepScores` column
sleepScores column is currently nested. `pd.json_normalize()` unpacks it into own columns and then we join those new columns back to the DataFrame

In [15]:
# Flatten 'sleepScores' column

scores_df = pd.json_normalize(df["sleepScores"].dropna())
scores_df.index = df["sleepScores"].dropna().index
scores_df.columns = ["score_" + col for col in scores_df.columns]
 
df = df.drop(columns=["sleepScores"]).join(scores_df)
 
print(f"Shape after flattening sleepScores: {df.shape}")
print(f"\nColumns:\n{list(df.columns)}")


Shape after flattening sleepScores: (800, 31)

Columns:
['sleepStartTimestampGMT', 'sleepEndTimestampGMT', 'calendarDate', 'sleepWindowConfirmationType', 'napList', 'deepSleepSeconds', 'lightSleepSeconds', 'remSleepSeconds', 'awakeSleepSeconds', 'unmeasurableSeconds', 'averageRespiration', 'lowestRespiration', 'highestRespiration', 'retro', 'awakeCount', 'avgSleepStress', 'restlessMomentCount', 'score_overallScore', 'score_qualityScore', 'score_durationScore', 'score_recoveryScore', 'score_deepScore', 'score_remScore', 'score_lightScore', 'score_awakeningsCountScore', 'score_awakeTimeScore', 'score_combinedAwakeScore', 'score_restfulnessScore', 'score_interruptionsScore', 'score_feedback', 'score_insight']


---
## 5. Sort by date
Rows with no date (empty nights) sort to the bottom.

In [16]:
df = df.sort_values("calendarDate").reset_index(drop=True)

print(f"\nDate range: {df['calendarDate'].iloc[0]}  →  {df['calendarDate'].iloc[-1]}")



Date range: 2023-12-11  →  nan


---
## Save to CSV

In [17]:
output_path = r"C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\data\processed\sleep_concat.csv"
df.to_csv(output_path, index=False)
 
print(f"\nSaved to: {output_path}")


Saved to: C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\data\processed\sleep_concat.csv
